<a href="https://colab.research.google.com/github/navacron/aistuff/blob/main/PartialInference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers peft accelerate

import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model


In [ ]:
base_model_name = "gpt2"   # or "tiiuae/falcon-7b-instruct" on a bigger GPU

config = AutoConfig.from_pretrained(base_model_name)
config.output_hidden_states = True  # crucial for grabbing layer-H activations

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    config=config,
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

3. Run once up to layer H and cache the activations

Here we treat layer index H in the Transformer block stack. For GPT-2, config.n_layer is the number of blocks.

In [25]:
import torch

H = 6  # for example

@torch.no_grad()
def forward_to_layer_H(input_ids, attention_mask=None, H=H):
    outputs = base_model.transformer(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        use_cache=False,
    )
    hidden_states = outputs.hidden_states
    hidden_H = hidden_states[H + 1]  # 0 = embeddings
    return hidden_H


You’d typically wrap this in a cache keyed by your input text / IDs:

In [26]:
hidden_cache = {}

def get_cached_hidden_H(text, H=H):
    if text in hidden_cache:
        return hidden_cache[text]

    enc = tokenizer(text, return_tensors="pt").to(base_model.device)
    hidden_H = forward_to_layer_H(enc["input_ids"], enc["attention_mask"], H=H)
    hidden_cache[text] = (hidden_H, enc["attention_mask"])
    return hidden_cache[text]


4. Build a “tail model” (layers H..end) with LoRA

Idea:

We reuse the base model’s first H blocks by calling forward_to_layer_H.

Then build a module that only runs blocks H..end + final LN + LM head.

We apply LoRA only to these tail blocks, so we can have multiple adapters there.

In [31]:
import torch.nn as nn

class GPT2Tail(nn.Module):
    def __init__(self, base_model, H):
        super().__init__()
        self.H = H
        self.config = base_model.config

        self.tail_blocks = nn.ModuleList(base_model.transformer.h[H:])
        self.ln_f = base_model.transformer.ln_f
        self.lm_head = base_model.lm_head

    def forward(self, hidden_states, attention_mask=None):
        x = hidden_states
        # DON'T pass attention_mask into the block; GPT-2 is causal-only
        for block in self.tail_blocks:
            x = block(x)[0]
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

    def prepare_inputs_for_generation(self, *args, **kwargs):
        raise NotImplementedError(
            "GPT2Tail does not support .generate(); "
            "call it directly with hidden states."
        )





In [24]:
import inspect
print(inspect.signature(GPT2Tail.forward))


(self, hidden_states, attention_mask=None)


Now wrap that tail with LoRA via PEFT:

In [33]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],
    bias="none",
    task_type="CAUSAL_LM",
    fan_in_fan_out=True,
)

tail_adapterA = get_peft_model(GPT2Tail(base_model, H=H), lora_config).to(base_model.device)
tail_adapterB = get_peft_model(GPT2Tail(base_model, H=H), lora_config).to(base_model.device)

tail_adapterA.print_trainable_parameters()




trainable params: 405,504 || all params: 81,531,648 || trainable%: 0.4974


5. Using multiple LoRA tails on the same cached activations

You can create multiple LoRA-augmented tails that all share the same underlying weights but have different adapters:

In [34]:
# Adapter A
#tail_adapterA = get_peft_model(GPT2Tail(base_model, H=H), lora_config).to(base_model.device)
# ... fine-tune tail_adapterA on task A, then save

# Adapter B
#tail_adapterB = get_peft_model(GPT2Tail(base_model, H=H), lora_config).to(base_model.device)
# ... fine-tune tail_adapterB on task B, then save


Infernece Time

In [43]:
@torch.no_grad()
def run_with_adapter(tail_lora, text, k=10):
    hidden_H, attention_mask = get_cached_hidden_H(text)

    # Call the *wrapped* GPT2Tail directly, WITHOUT attention_mask
    logits = tail_lora.model(hidden_H)

    last_logits = logits[:, -1, :]
    probs = last_logits.softmax(dim=-1)
    topk = torch.topk(probs, k)
    tokens = [tokenizer.decode([i]) for i in topk.indices[0].tolist()]
    return list(zip(tokens, topk.values[0].tolist()))





In [51]:
print("Adapter A:")
print(run_with_adapter(tail_adapterA, "The capital of France is"))

print("Adapter B:")
print(run_with_adapter(tail_adapterB, "The capital of France is"))


Adapter A:
[(' Paris', 0.94091796875), (' Berlin', 0.0565185546875), (' Nice', 0.0007114410400390625), (' Brussels', 0.0004591941833496094), (' France', 0.00029659271240234375), (' Marse', 0.00010246038436889648), (' Stras', 5.4836273193359375e-05), (' Le', 5.4836273193359375e-05), (' B', 4.547834396362305e-05), (' Lyon', 4.547834396362305e-05)]
Adapter B:
[(' Paris', 0.94091796875), (' Berlin', 0.0565185546875), (' Nice', 0.0007114410400390625), (' Brussels', 0.0004591941833496094), (' France', 0.00029659271240234375), (' Marse', 0.00010246038436889648), (' Stras', 5.4836273193359375e-05), (' Le', 5.4836273193359375e-05), (' B', 4.547834396362305e-05), (' Lyon', 4.547834396362305e-05)]


Run full model for sanity test

In [46]:
@torch.no_grad()
def debug_compare(prompt, k=10):
    enc = tokenizer(prompt, return_tensors="pt").to(base_model.device)

    # Full model logits
    logits_full = base_model(**enc).logits[:, -1, :]
    probs_full = logits_full.softmax(dim=-1)
    topk_full = torch.topk(probs_full, k)

    # Trunk + tail (Adapter A, but LoRA is near-zero so it's basically base tail)
    hidden_H = forward_to_layer_H(enc["input_ids"], enc["attention_mask"])
    logits_tail = tail_adapterA.model(hidden_H)
    probs_tail = logits_tail[:, -1, :].softmax(dim=-1)
    topk_tail = torch.topk(probs_tail, k)

    def decode_topk(topk):
        return [(tokenizer.decode([i]), float(v)) for i, v in zip(topk.indices[0], topk.values[0])]

    print("FULL MODEL TOP-k:")
    print(decode_topk(topk_full))
    print("\nTRUNK+TAIL TOP-k:")
    print(decode_topk(topk_tail))

debug_compare("The capital of Japan is", k=10)


FULL MODEL TOP-k:
[(' the', 0.0941162109375), (' Tokyo', 0.06884765625), (' home', 0.064697265625), (' a', 0.041778564453125), (' now', 0.03924560546875), (' located', 0.028717041015625), (' known', 0.0269775390625), (' Japan', 0.0253448486328125), (' in', 0.0197296142578125), (' also', 0.0174102783203125)]

TRUNK+TAIL TOP-k:
[(' the', 0.108154296875), (' home', 0.0791015625), (' Tokyo', 0.06561279296875), (' now', 0.0479736328125), (' a', 0.042327880859375), (' located', 0.0290985107421875), (' Japan', 0.02734375), (' in', 0.0226593017578125), (' known', 0.0212860107421875), (' also', 0.0165863037109375)]


Now lets fine tune both lora

In [47]:
import torch
import torch.nn.functional as F

device = base_model.device  # just to be explicit

# Toy "France expert" adapter
train_A = [
    ("The capital of France is", " Paris"),
    ("In Europe, the capital of France is", " Paris"),
    ("France's capital city is", " Paris"),
]

# Toy "Germany expert" adapter
train_B = [
    ("The capital of Germany is", " Berlin"),
    ("In Europe, the capital of Germany is", " Berlin"),
    ("Germany's capital city is", " Berlin"),
]


train adapter

In [48]:
def finetune_adapter(tail_adapter, train_data, epochs=50, lr=5e-4):
    tail_adapter.train()
    optimizer = torch.optim.AdamW(tail_adapter.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0.0

        for prompt, target_token_str in train_data:
            # 1) Get trunk output (cached or computed)
            hidden_H, _ = get_cached_hidden_H(prompt)  # hidden_H on correct device

            # 2) Forward through LoRA tail
            logits = tail_adapter.model(hidden_H)  # (batch=1, seq, vocab)
            last_logits = logits[:, -1, :]        # next-token prediction

            # 3) Build target id for the desired next token
            target_id = tokenizer.encode(
                target_token_str, add_special_tokens=False
            )[0]
            target = torch.tensor([target_id], device=last_logits.device)

            # 4) Cross-entropy loss and step
            loss = F.cross_entropy(last_logits, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            avg_loss = total_loss / len(train_data)
            print(f"Epoch {epoch+1}: loss = {avg_loss:.4f}")

    tail_adapter.eval()


In [49]:
print("Training Adapter A (France → Paris)...")
finetune_adapter(tail_adapterA, train_A, epochs=60)

print("\nTraining Adapter B (Germany → Berlin)...")
finetune_adapter(tail_adapterB, train_B, epochs=60)


Training Adapter A (France → Paris)...
Epoch 10: loss = 0.0000
Epoch 20: loss = 0.0000
Epoch 30: loss = 0.0000
Epoch 40: loss = 0.0000
Epoch 50: loss = 0.0000
Epoch 60: loss = 0.0000

Training Adapter B (Germany → Berlin)...
Epoch 10: loss = 0.0145
Epoch 20: loss = 0.0026
Epoch 30: loss = 0.0013
Epoch 40: loss = 0.0007
Epoch 50: loss = 0.0003
Epoch 60: loss = 0.0013


Now run the two prompts, see if Paris is fine tuned with higher probability and Berlin for second adapter.

In [50]:
print("=== Prompt: The capital of France is ===")
print("Adapter A:", run_with_adapter(tail_adapterA, "The capital of France is"))
print("Adapter B:", run_with_adapter(tail_adapterB, "The capital of France is"))

print("\n=== Prompt: The capital of Germany is ===")
print("Adapter A:", run_with_adapter(tail_adapterA, "The capital of Germany is"))
print("Adapter B:", run_with_adapter(tail_adapterB, "The capital of Germany is"))


=== Prompt: The capital of France is ===
Adapter A: [(' Paris', 0.94091796875), (' Berlin', 0.0565185546875), (' Nice', 0.0007114410400390625), (' Brussels', 0.0004591941833496094), (' France', 0.00029659271240234375), (' Marse', 0.00010246038436889648), (' Stras', 5.4836273193359375e-05), (' Le', 5.4836273193359375e-05), (' B', 4.547834396362305e-05), (' Lyon', 4.547834396362305e-05)]
Adapter B: [(' Paris', 0.94091796875), (' Berlin', 0.0565185546875), (' Nice', 0.0007114410400390625), (' Brussels', 0.0004591941833496094), (' France', 0.00029659271240234375), (' Marse', 0.00010246038436889648), (' Stras', 5.4836273193359375e-05), (' Le', 5.4836273193359375e-05), (' B', 4.547834396362305e-05), (' Lyon', 4.547834396362305e-05)]

=== Prompt: The capital of Germany is ===
Adapter A: [(' Berlin', 1.0), (' Munich', 1.4722347259521484e-05), (' Cologne', 1.4722347259521484e-05), (' Hamburg', 1.2993812561035156e-05), (' Frankfurt', 1.2993812561035156e-05), (' Germany', 5.781650543212891e-06), 